# Currency Exchange Pipeline Demo
This notebook demonstrates the execution of the ETL pipeline and inspects the data in Bronze, Silver, and Gold layers.

In [1]:
import os
import sys
import pandas as pd

# Add project root to path
sys.path.append(os.path.abspath('..'))

from pipeline.utils import get_db_connection
from pipeline.scheduler import run_pipeline

ModuleNotFoundError: No module named 'load_bronze'

## 1. Run the Pipeline
We will run the pipeline for a small date range.

In [ ]:
run_pipeline(start_date='2024-01-26', end_date='2024-01-30')

## 2. Inspect Bronze Layer
Raw JSON data as received from the API.

In [ ]:
conn = get_db_connection()
pd.read_sql('SELECT * FROM bronze.raw_rates ORDER BY fetch_date DESC LIMIT 5', conn)

## 3. Inspect Silver Layer
Structured relational data.

In [ ]:
pd.read_sql('SELECT * FROM silver.cleaned_rates ORDER BY date DESC LIMIT 10', conn)

## 4. Inspect Gold Layer
Analytical Fact table joined with Dimensions.

In [ ]:
query = '''
    SELECT f.date, d1.currency_code as base, d2.currency_code as target, f.exchange_rate
    FROM gold.fact_exchange_rates f
    JOIN gold.dim_currency d1 ON f.base_currency = d1.currency_code
    JOIN gold.dim_currency d2 ON f.target_currency = d2.currency_code
    ORDER BY f.date DESC
    LIMIT 10
'''
pd.read_sql(query, conn)